# Preparação dos Dados

A ideia aqui é criar dataframes que vão ajudar legal nos sistemas de recomendação.
A abordagem escolhida nesse ponto foi a abordagem por filtragem hibrida. Nos pretendemos usar filtragem colaborativa para encontrar pessoas que tem distribuições de dinheiro em investimentos parecidas entre si. A partir disso, vamos verificar quais ativos essas pessoas de perfil parecido também possuem, então faremos filtragem por item, para encontrar relações entre os itens que as pessoas com pensamento de investimento parecidas com o alvo possuem e recomendar uma tendencia para o alvo.

In [1]:
import pandas as pd
import numpy as np

In [2]:
filepath = '../data/processed/dataframe_classificado.csv'

df = pd.read_csv(filepath)

print(f"Dimensões do DataFrame: {df.shape}")
df.head()

Dimensões do DataFrame: (286146, 8)


,CONTA,FINANCEIRO,QUANTIDADE,NOME ATIVO,TIPO DE ATIVO,PERFIL DA CARTEIRA,CLASSIFICACAO DO ATIVO,Subgrupo
0,27,16278.264089,16.0,CRI CDIE,TITULOS RF PRIVADOS BRASIL,Moderado,Renda Fixa,RF - Títulos Privados Nacionais
1,27,25081.618073,25.0,CRA CDIE,TITULOS RF PRIVADOS BRASIL,Moderado,Renda Fixa,RF - Títulos Privados Nacionais
2,27,47046.248881,49.0,CRA IPCA,TITULOS RF PRIVADOS BRASIL,Moderado,Renda Fixa,RF - Títulos Privados Nacionais
3,27,36331.476685,35.0,RRRP13 IPCA,DEBENTURES BRASIL,Moderado,Renda Fixa,RF - Títulos Privados Nacionais
4,27,18995.623640,20.0,EEELA1 IPCA,DEBENTURES BRASIL,Moderado,Renda Fixa,RF - Títulos Privados Nacionais


Aqui minha ideia é descobrir quanto de dinheiro cada uma das contas colocou em cada classificação de ativo e cada subgrupo pra tentar traçar um perfil de investidor e classificar ele em um dos 3 grupos (conservador, moderado ou sofisticado)

In [3]:
columns_to_drop = ['QUANTIDADE', 'TIPO DE ATIVO', 'NOME ATIVO', 'PERFIL DA CARTEIRA']
df_droped = df.drop(columns_to_drop, axis=1)
df_droped.head()

,CONTA,FINANCEIRO,CLASSIFICACAO DO ATIVO,Subgrupo
0,27,16278.264089,Renda Fixa,RF - Títulos Privados Nacionais
1,27,25081.618073,Renda Fixa,RF - Títulos Privados Nacionais
2,27,47046.248881,Renda Fixa,RF - Títulos Privados Nacionais
3,27,36331.476685,Renda Fixa,RF - Títulos Privados Nacionais
4,27,18995.623640,Renda Fixa,RF - Títulos Privados Nacionais


Sabendo que investimentos negativos significam que o alvo entrou vendido, então eu considero o módulo do valor financeiro, e assim eu dou groupby no valor financeiro por conta e classificação do ativo, pra saber quanto cada usuário ja investiu em renda fixa e renda variavel

In [4]:
df_temp = df_droped.copy()
df_temp['FINANCEIRO'] = df_temp['FINANCEIRO'].abs()

print(f"Number of negative values in original FINANCEIRO column: {(df['FINANCEIRO'] < 0).sum()}")
print(f"Number of negative values in df_temp FINANCEIRO column: {(df_temp['FINANCEIRO'] < 0).sum()}")

df_grouped = df_temp.groupby(['CONTA', 'CLASSIFICACAO DO ATIVO'])['FINANCEIRO'].sum().reset_index()

print(f"Shape of grouped DataFrame: {df_grouped.shape}")
df_grouped

Number of negative values in original FINANCEIRO column: 18974
Number of negative values in df_temp FINANCEIRO column: 0
Shape of grouped DataFrame: (7906, 3)


,CONTA,CLASSIFICACAO DO ATIVO,FINANCEIRO
0,27,Renda Fixa,2.993216e+06
1,44,Renda Fixa,1.700028e+06
2,103,Renda Fixa,8.604886e+06
3,285,Renda Fixa,1.339570e+06
4,299,Renda Fixa,5.453470e+05
...,...,...,...
7901,6150189,Renda Variável,1.631883e+08
7902,6167502,Renda Fixa,6.104810e+08
7903,6178808,Renda Fixa,2.055990e+07
7904,6184909,Renda Fixa,5.002625e+05


Aqui o ponto é transformar esses valores de investimento em Renda Fixa e Renda Variavel em percentuais de investimento, assim uma pessoa que investe 1 milhão e investem 10 mil ainda podem se assemelhar a nível de tipo de investidor

In [ ]:
total_by_account = df_grouped.groupby('CONTA')['FINANCEIRO'].sum().reset_index()
total_by_account.rename(columns={'FINANCEIRO': 'TOTAL_FINANCEIRO'}, inplace=True)

df_with_total = pd.merge(df_grouped, total_by_account, on='CONTA', how='left')

df_with_total['PERCENTUAL'] = df_with_total['FINANCEIRO'] / df_with_total['TOTAL_FINANCEIRO'] * 100

df_result = df_with_total.pivot_table(
    index='CONTA', 
    columns='CLASSIFICACAO DO ATIVO', 
    values='PERCENTUAL', 
    fill_value=0
)

required_columns = ['Renda Fixa', 'Renda Variável']
for col in required_columns:
    if col not in df_result.columns:
        df_result[col] = 0

df_result = df_result[required_columns]

df_result.columns.name = None  
df_result.index.name = 'Conta'  

print(f"Shape of new DataFrame: {df_result.shape}")
df_result.head()


Shape of new DataFrame: (6101, 2)


,Renda Fixa,Renda Variável
Conta,,
27,100.0,0.0
44,100.0,0.0
103,100.0,0.0
285,100.0,0.0
299,100.0,0.0


In [ ]:
df_final = df_result.reset_index()  

df_final = df_final[['Conta', 'Renda Fixa', 'Renda Variável']]

print(f"Shape of final DataFrame: {df_final.shape}")
df_final.head(100)


Shape of final DataFrame: (6101, 3)


,Conta,Renda Fixa,Renda Variável
0,27,100.000000,0.000000
1,44,100.000000,0.000000
2,103,100.000000,0.000000
3,285,100.000000,0.000000
4,299,100.000000,0.000000
...,...,...,...
95,16524,100.000000,0.000000
96,16759,98.710961,1.289039
97,16800,100.000000,0.000000
98,16821,100.000000,0.000000


In [7]:
df_final.to_csv('../data/processed/percentual_de_distribuicao.csv', index=False)

Depois de baixar esse dado precioso, vamos para próxima mina de ouro: Descobrir os percentuais de investimento em cada subgrupo.
De primeira, nossa abordagem envolveria a filtragem colaborativa por usuário pegando semelhanças em investimemnto verificando suas distribuições em renda fixa e renda variável. Mas depois de testes, percebemos que dentro do mundo de renda fixa e variavel existem diversos tipos de investimento que qualificam diferentes tipos de investidores (uma pessoa que investe em commodite é diferente de uma pessoa que investe em câmbio, por exemplo).

In [8]:
df

,CONTA,FINANCEIRO,QUANTIDADE,NOME ATIVO,TIPO DE ATIVO,PERFIL DA CARTEIRA,CLASSIFICACAO DO ATIVO,Subgrupo
0,27,1.627826e+04,16.0,CRI CDIE,TITULOS RF PRIVADOS BRASIL,Moderado,Renda Fixa,RF - Títulos Privados Nacionais
1,27,2.508162e+04,25.0,CRA CDIE,TITULOS RF PRIVADOS BRASIL,Moderado,Renda Fixa,RF - Títulos Privados Nacionais
2,27,4.704625e+04,49.0,CRA IPCA,TITULOS RF PRIVADOS BRASIL,Moderado,Renda Fixa,RF - Títulos Privados Nacionais
3,27,3.633148e+04,35.0,RRRP13 IPCA,DEBENTURES BRASIL,Moderado,Renda Fixa,RF - Títulos Privados Nacionais
4,27,1.899562e+04,20.0,EEELA1 IPCA,DEBENTURES BRASIL,Moderado,Renda Fixa,RF - Títulos Privados Nacionais
...,...,...,...,...,...,...,...,...
286141,6178808,3.890751e+05,400.0,RSAN16 IPCA,DEBENTURES BRASIL,Sofisticado,Renda Fixa,RF - Títulos Privados Nacionais
286142,6178808,4.049765e+05,423.0,CMGDB1 IPCA,DEBENTURES BRASIL,Sofisticado,Renda Fixa,RF - Títulos Privados Nacionais
286143,6178808,1.576249e+06,1557.0,EGIEB4 IPCA,DEBENTURES BRASIL,Sofisticado,Renda Fixa,RF - Títulos Privados Nacionais
286144,6184909,5.002625e+05,637.0,NTNF PRE,TITULOS RF PUBLICOS BRASIL,Conservador,Renda Fixa,RF - Títulos Públicos Nacionais


In [ ]:
df_subgroup = df_temp.copy()

subgroup_grouped = df_subgroup.groupby(['CONTA', 'Subgrupo'])['FINANCEIRO'].sum().reset_index()

subgroup_total = subgroup_grouped.groupby('CONTA')['FINANCEIRO'].sum().reset_index()
subgroup_total.rename(columns={'FINANCEIRO': 'TOTAL_FINANCEIRO'}, inplace=True)

subgroup_with_total = pd.merge(subgroup_grouped, subgroup_total, on='CONTA', how='left')

subgroup_with_total['PERCENTUAL'] = subgroup_with_total['FINANCEIRO'] / subgroup_with_total['TOTAL_FINANCEIRO'] * 100

subgroup_result = subgroup_with_total.pivot_table(
    index='CONTA',
    columns='Subgrupo',
    values='PERCENTUAL',
    fill_value=0
)

subgroup_final = subgroup_result.reset_index()

print(f"Shape of subgroup distribution DataFrame: {subgroup_final.shape}")
subgroup_final.head()



Shape of subgroup distribution DataFrame: (6101, 15)


Subgrupo,CONTA,RF - Derivativos de Renda Fixa,RF - Títulos Internacionais,RF - Títulos Privados Nacionais,RF - Títulos Públicos Nacionais,RV - Ações Internacionais e ADRs,RV - Ações Nacionais,RV - Commodities (Spot),RV - Derivativos - Futuros de Ações Individuais,RV - Derivativos - Futuros de Commodities,RV - Derivativos - Futuros de Moedas,RV - Derivativos - Futuros de Taxas de Juros,RV - Derivativos - Futuros de Índice,RV - Derivativos - Opções,RV - Moedas (Spot)
0,27,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,44,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,103,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,285,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,299,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
subgroup_final.to_csv('../data/processed/subgrupo_distribuicao.csv', index=False)

Guardamos esse dado para fazer filtragens futuras. O novo objetivo é plotar uma matriz onde cada coluna é um ativo e cada linha é a representação de quais ativos cada conta possui. Fazemos isso pensando em uma filtragem por item futuramente!

In [11]:
user_item_binary = df.assign(tem_ativo=1).pivot_table(
    index='CONTA',
    columns='NOME ATIVO', 
    values='tem_ativo',
    aggfunc='max',
    fill_value=0
).reset_index()

user_item_binary.head()


NOME ATIVO,CONTA,1832421D CN,1SANPN,2299955D CN,2458693D US,700 HK,8001 JP,8031 JP,8058 JP,9988 HK,...,YUFI11,Z US.,Z1OM34,ZAGH11,ZAMP3,ZAVI11,ZIFI11,ZS US,ZTO US,ZTS US
0,27,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,44,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,103,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,285,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,299,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [12]:
user_item_binary.to_csv('../data/processed/user_item_binario.csv', index=False)